# NLLB-200 Fine-Tuning: English → Twi Translation

This notebook fine-tunes `facebook/nllb-200-distilled-600M` for English → Twi translation.

**Model:** NLLB-200 (600M params, distilled)  
**Dataset:** GhanaNLP Twi-English Parallel Text  
**Platform:** Kaggle P100 (16 GB VRAM)  
**Direction:** `eng_Latn` → `twi_Latn`

## Why a Separate En→Twi Model?
- v1 showed significant BLEU gap: Twi→En (10.50) vs En→Twi (6.32)
- Direction-separated models eliminate confusion from mixed training
- Each model specializes in its specific translation direction

## 1. Install Dependencies

In [ ]:
!pip install -q transformers datasets sacrebleu sentencepiece \
             accelerate evaluate scikit-learn langdetect

## 2. Configuration

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CONFIGURATION — English → Twi Direction
# ═══════════════════════════════════════════════════════════════════════════

# Model configuration
MODEL_ID = "facebook/nllb-200-distilled-600M"
SRC_LANG = "eng_Latn"   # English (source)
TGT_LANG = "twi_Latn"   # Twi (target)
MAX_LEN = 256

# Training configuration
NUM_EPOCHS = 10
BATCH_SIZE = 8
GRAD_ACCUM = 4
LEARNING_RATE = 3e-5
WARMUP_RATIO = 0.06
WEIGHT_DECAY = 0.01
LABEL_SMOOTHING = 0.1

# Output
OUTPUT_DIR = "./en-twi-nllb-v2"
HF_REPO = "ninte/en-twi-nllb-v2"  # Change to your HF username

print("Configuration loaded.")
print(f"  Model: {MODEL_ID}")
print(f"  Direction: {SRC_LANG} → {TGT_LANG}")
print(f"  Effective batch size: {BATCH_SIZE * GRAD_ACCUM}")

## 3. Load and Clean Dataset

In [ ]:
from datasets import load_dataset
import pandas as pd
import re
import warnings
warnings.filterwarnings('ignore')

# Load BOTH datasets from HF Hub
print("Loading datasets from HuggingFace Hub...")
ds1 = load_dataset("Ghana-NLP/TWI_ENGLISH_PARALLEL_TEXT")
ds2 = load_dataset("Ghana-NLP/ENGLISH_TWI_PARALLEL_TEXT")

# Convert and standardize
df1 = ds1['train'].to_pandas().rename(columns={'text': 'twi', 'label': 'english'})
df2 = ds2['train'].to_pandas().rename(columns={'text': 'english', 'label': 'twi'})

# Combine
df = pd.concat([
    df1[['twi', 'english']],
    df2[['twi', 'english']]
], ignore_index=True)

print(f"Combined raw: {len(df):,} pairs")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# DATA CLEANING
# ═══════════════════════════════════════════════════════════════════════════

TWI_SPECIAL = set('ɛɔŋƐƆŊ')

def clean_text(text):
    if not isinstance(text, str) or pd.isna(text):
        return None
    text = re.sub(r'^\d+\.\s*', '', text.strip())
    text = re.sub(r' +', ' ', text)
    return text.strip() or None

def is_twi_text(text):
    return any(c in TWI_SPECIAL for c in str(text))

# Clean
df['twi'] = df['twi'].apply(clean_text)
df['english'] = df['english'].apply(clean_text)
df = df.dropna(subset=['twi', 'english'])
print(f"After null removal: {len(df):,}")

# Word count filter
df['twi_wc'] = df['twi'].str.split().str.len()
df['eng_wc'] = df['english'].str.split().str.len()
df = df[(df['twi_wc'] >= 3) & (df['eng_wc'] >= 3)]
df = df[(df['twi_wc'] <= 150) & (df['eng_wc'] <= 150)]
print(f"After length filter: {len(df):,}")

# Remove direction-swapped pairs
df = df[df['twi'].apply(lambda x: not all(ord(c) < 128 for c in str(x).replace(' ', '')))]
print(f"After swap removal: {len(df):,}")

# Deduplicate
df = df.drop_duplicates(subset=['twi', 'english'])
df = df.drop(columns=['twi_wc', 'eng_wc']).reset_index(drop=True)
print(f"After deduplication: {len(df):,}")

## 4. Train/Val/Test Split

In [ ]:
from sklearn.model_selection import train_test_split

# 80/10/10 split
train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)

print(f"Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}")

## 5. Load NLLB Model and Tokenizer

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

print(f"Loading {MODEL_ID}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_ID)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

print(f"Device: {device}")
print(f"Model params: {sum(p.numel() for p in model.parameters())/1e6:.1f}M")
if torch.cuda.is_available():
    print(f"VRAM used: {torch.cuda.memory_allocated()/1e9:.2f} GB")

## 6. Tokenize Dataset (English → Twi)

In [ ]:
from datasets import Dataset

def tokenize_batch_en_to_twi(batch):
    """
    Tokenize a batch for NLLB (English → Twi direction).
    Source: English, Target: Twi
    """
    # Set source language to ENGLISH
    tokenizer.src_lang = SRC_LANG  # eng_Latn
    
    # Tokenize source (English)
    model_inputs = tokenizer(
        batch["english"],  # Source is English!
        max_length=MAX_LEN,
        truncation=True,
        padding="max_length",
    )
    
    # Tokenize target (Twi)
    labels = tokenizer(
        text_target=batch["twi"],  # Target is Twi!
        max_length=MAX_LEN,
        truncation=True,
        padding="max_length",
    )
    
    # Mask padding in labels
    labels["input_ids"] = [
        [(t if t != tokenizer.pad_token_id else -100) for t in label]
        for label in labels["input_ids"]
    ]
    model_inputs["labels"] = labels["input_ids"]
    
    return model_inputs

# Convert to HuggingFace datasets
train_ds = Dataset.from_pandas(train_df[['twi', 'english']].reset_index(drop=True))
val_ds = Dataset.from_pandas(val_df[['twi', 'english']].reset_index(drop=True))
test_ds = Dataset.from_pandas(test_df[['twi', 'english']].reset_index(drop=True))

# Tokenize
print("Tokenizing datasets (English → Twi)...")
train_tok = train_ds.map(tokenize_batch_en_to_twi, batched=True, batch_size=256,
                          remove_columns=['twi', 'english'])
val_tok = val_ds.map(tokenize_batch_en_to_twi, batched=True, batch_size=256,
                      remove_columns=['twi', 'english'])
test_tok = test_ds.map(tokenize_batch_en_to_twi, batched=True, batch_size=256,
                        remove_columns=['twi', 'english'])

print("Tokenization complete.")

## 7. Define Evaluation Metrics

In [ ]:
import evaluate
import numpy as np

bleu_metric = evaluate.load("sacrebleu")
chrf_metric = evaluate.load("chrf")
ter_metric = evaluate.load("ter")

def compute_metrics(eval_preds):
    """Compute BLEU, chrF, TER for En→Twi."""
    preds, labels = eval_preds
    
    if isinstance(preds, tuple):
        preds = preds[0]
    
    preds = np.argmax(preds, axis=-1) if preds.ndim == 3 else preds
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    
    bleu = bleu_metric.compute(
        predictions=decoded_preds,
        references=[[r] for r in decoded_labels]
    )
    chrf = chrf_metric.compute(
        predictions=decoded_preds,
        references=decoded_labels
    )
    ter = ter_metric.compute(
        predictions=decoded_preds,
        references=[[r] for r in decoded_labels]
    )
    
    return {
        "bleu": round(bleu["score"], 2),
        "chrf": round(chrf["score"], 2),
        "ter": round(ter["score"], 2),
    }

print("Metrics ready: BLEU, chrF, TER")

## 8. Training

In [ ]:
from transformers import (
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq,
)
import os

data_collator = DataCollatorForSeq2Seq(
    tokenizer,
    model=model,
    label_pad_token_id=-100,
    pad_to_multiple_of=8,
)

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    warmup_ratio=WARMUP_RATIO,
    weight_decay=WEIGHT_DECAY,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    predict_with_generate=True,
    generation_max_length=MAX_LEN,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="chrf",  # chrF is primary for Twi
    greater_is_better=True,
    logging_steps=50,
    fp16=True,
    dataloader_num_workers=2,
    report_to="none",
    label_smoothing_factor=LABEL_SMOOTHING,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print(f"Trainer initialized for English → Twi")

In [ ]:
# Checkpoint resume
checkpoints = []
if os.path.exists(OUTPUT_DIR):
    checkpoints = [f for f in os.listdir(OUTPUT_DIR) if f.startswith("checkpoint-")]

if checkpoints:
    latest = sorted(checkpoints, key=lambda x: int(x.split("-")[-1]))[-1]
    resume_from = os.path.join(OUTPUT_DIR, latest)
    print(f"Resuming from: {resume_from}")
else:
    resume_from = None
    print("Starting fresh training run")

# Train!
trainer.train(resume_from_checkpoint=resume_from)

## 9. Full Evaluation

In [ ]:
import sacrebleu as sb

def full_evaluate_en_to_twi(model, tokenizer, df, label=''):
    """
    Full evaluation for English → Twi direction.
    """
    model.eval()
    tgt_token_id = tokenizer.convert_tokens_to_ids(TGT_LANG)  # twi_Latn
    preds, refs = [], []
    
    for _, row in df.iterrows():
        tokenizer.src_lang = SRC_LANG  # eng_Latn
        inputs = tokenizer(
            row['english'],  # Source is English
            return_tensors="pt",
            truncation=True,
            max_length=MAX_LEN
        )
        inputs = {k: v.to(model.device) for k, v in inputs.items()}
        
        with torch.no_grad():
            out = model.generate(
                **inputs,
                forced_bos_token_id=tgt_token_id,
                num_beams=4,
                max_length=MAX_LEN,
                no_repeat_ngram_size=3,
                length_penalty=1.0,
                early_stopping=True,
            )
        
        preds.append(tokenizer.decode(out[0], skip_special_tokens=True))
        refs.append(row['twi'])  # Reference is Twi
    
    bleu = sb.corpus_bleu(preds, [refs])
    chrf = sb.corpus_chrf(preds, [refs])
    ter = sb.corpus_ter(preds, [refs])
    
    print(f"\n{'='*55}")
    print(f"  {label} ({len(df)} pairs)")
    print(f"{'='*55}")
    print(f"  BLEU   : {bleu.score:.2f}")
    print(f"  chrF   : {chrf.score:.2f}")
    print(f"  TER    : {ter.score:.2f}  (lower is better)")
    print(f"  BLEU 1g: {bleu.precisions[0]:.2f}")
    print(f"  BLEU 4g: {bleu.precisions[3]:.2f}")
    print(f"  BP     : {bleu.bp:.4f}")
    print(f"{'='*55}\n")
    
    return {"bleu": bleu.score, "chrf": chrf.score, "ter": ter.score,
            "preds": preds, "refs": refs}

In [ ]:
# Full test evaluation
print("Running full evaluation on test set...")
results = full_evaluate_en_to_twi(
    model, tokenizer, test_df,
    label="English → Twi Test Set"
)

# v1 baseline comparison
print("\n" + "="*55)
print("  v1 BASELINE COMPARISON (En→Twi)")
print("="*55)
print(f"  v1 En→Twi: BLEU 6.32 | chrF 28.74 | TER 91.55")
print(f"  v2 En→Twi: BLEU {results['bleu']:.2f} | chrF {results['chrf']:.2f} | TER {results['ter']:.2f}")
improvement = results['bleu'] - 6.32
print(f"  BLEU improvement: {improvement:+.2f}")
print("="*55)

## 10. Translation Examples

In [ ]:
def translate_en_to_twi(text):
    """
    Translate English text to Twi.
    """
    tokenizer.src_lang = SRC_LANG  # eng_Latn
    tgt_token_id = tokenizer.convert_tokens_to_ids(TGT_LANG)  # twi_Latn
    
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_LEN
    )
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    
    with torch.no_grad():
        out = model.generate(
            **inputs,
            forced_bos_token_id=tgt_token_id,
            num_beams=4,
            max_length=MAX_LEN,
            no_repeat_ngram_size=3,
            length_penalty=1.0,
            early_stopping=True,
        )
    
    return tokenizer.decode(out[0], skip_special_tokens=True)

# Test translations
test_sentences = [
    "The police are responsible for protecting citizens.",
    "This has led to a growing interest in education.",
    "The farmer cultivates maize and cocoa on his land.",
    "Good health is very important for everyone.",
]

print("\n" + "="*70)
print("TRANSLATION EXAMPLES (English → Twi)")
print("="*70)
for en in test_sentences:
    print(f"\nEnglish: {en}")
    print(f"Twi    : {translate_en_to_twi(en)}")

## 11. Push to HuggingFace Hub

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
# Update model config
model.config.update({
    "src_lang": SRC_LANG,  # eng_Latn
    "tgt_lang": TGT_LANG,  # twi_Latn
    "language": ["en", "tw"],
    "tags": ["translation", "twi", "akan", "ghana", "low-resource", "nllb", "en-tw"],
    "license": "cc-by-nc-4.0",
})

# Push to Hub
print(f"Pushing to {HF_REPO}...")
model.push_to_hub(HF_REPO)
tokenizer.push_to_hub(HF_REPO)

print(f"\nModel published: https://huggingface.co/{HF_REPO}")

## 12. Training Summary

In [ ]:
print("\n" + "="*70)
print("TRAINING SUMMARY — English → Twi")
print("="*70)
print(f"  Base model     : {MODEL_ID}")
print(f"  Direction      : {SRC_LANG} → {TGT_LANG}")
print(f"  Dataset        : GhanaNLP Twi-English Parallel Text")
print(f"  Training pairs : {len(train_df):,}")
print(f"  Validation     : {len(val_df):,}")
print(f"  Test pairs     : {len(test_df):,}")
print(f"  Epochs         : {NUM_EPOCHS}")
print(f"  Batch size     : {BATCH_SIZE} (effective: {BATCH_SIZE * GRAD_ACCUM})")
print(f"  Learning rate  : {LEARNING_RATE}")
print(f"\n  Final test BLEU: {results['bleu']:.2f}")
print(f"  Final test chrF: {results['chrf']:.2f}")
print(f"  Final test TER : {results['ter']:.2f}")
print(f"\n  Published to   : https://huggingface.co/{HF_REPO}")
print("="*70)